# 01 - Prefetch assets (ONLINE)

Run this **once**, on any accelerator, with **Internet ON**.
It vendors everything the offline run needs into `/kaggle/working`, which you then
attach to notebook 02 as a dataset.

| Asset | Size | Why it is needed |
|---|---|---|
| This project's repo | small | the `mlrx` package |
| NVlabs/edm | small | the pretrained `.pkl` cannot be unpickled without `torch_utils`, `dnnlib`, `training.networks` |
| EDM CIFAR-10 cond checkpoint | 213 MB | the denoiser |
| Inception-v3 (StyleGAN3 port) | 91 MB | FID features; torchvision's Inception gives different numbers |
| CIFAR-10 FID reference stats | 32 MB | the reference mu/sigma |
| pip wheels | small | anything missing from the offline image |

Total roughly **340 MB**.


In [ ]:
REPO_URL = 'https://github.com/azraihan/multi-level-richardson-faster-diffusion-ode'
REPO_BRANCH = 'main'          # or a tag/commit for a reproducible pin
EDM_COMMIT  = 'main'          # NVlabs/edm

OUT = '/kaggle/working'

ASSETS = {
    'edm-cifar10-32x32-cond-vp.pkl':
        'https://nvlabs-fi-cdn.nvidia.com/edm/pretrained/edm-cifar10-32x32-cond-vp.pkl',
    'cifar10-32x32.npz':
        'https://nvlabs-fi-cdn.nvidia.com/edm/fid-refs/cifar10-32x32.npz',
    'inception-2015-12-05.pkl':
        'https://api.ngc.nvidia.com/v2/models/nvidia/research/stylegan3/versions/1/'
        'files/metrics/inception-2015-12-05.pkl',
}

# Present in Kaggle's image already, but pinned here so the offline notebook
# never has to reach the network even if the base image changes.
WHEELS = ['click', 'tqdm', 'requests', 'psutil', 'imageio', 'pyspng']


In [ ]:
import os, subprocess, sys, time, json, hashlib, urllib.request

def sh(cmd, **kw):
    print('$', cmd, flush=True)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True, **kw)
    if r.stdout.strip(): print(r.stdout.strip()[:2000])
    if r.returncode != 0:
        print(r.stderr.strip()[:4000]); raise RuntimeError(f'failed: {cmd}')
    return r

for d in ('repo', 'third_party', 'assets', 'wheels'):
    os.makedirs(f'{OUT}/{d}', exist_ok=True)
print('workspace ready')


## Clone the sources

In [ ]:
sh(f'git clone --depth 1 --branch {REPO_BRANCH} {REPO_URL} {OUT}/repo/project')
sh(f'git clone --depth 1 https://github.com/NVlabs/edm.git {OUT}/third_party/edm')

# The pickle needs these three importable; fail loudly now rather than
# halfway through the offline run.
for m in ('torch_utils', 'dnnlib', 'training'):
    assert os.path.isdir(f'{OUT}/third_party/edm/{m}'), f'missing {m} in NVlabs/edm'
print('\nboth repos present, edm has torch_utils / dnnlib / training')


## Download the model, the FID detector, and the reference statistics

In [ ]:
def fetch(url, dest, retries=3):
    if os.path.exists(dest) and os.path.getsize(dest) > 0:
        print(f'  cached  {os.path.basename(dest)} '
              f'({os.path.getsize(dest)/2**20:.1f} MB)'); return dest
    for k in range(retries):
        try:
            t0 = time.time()
            with urllib.request.urlopen(url, timeout=120) as r, open(dest, 'wb') as f:
                total = int(r.headers.get('content-length', 0)); done = 0
                while True:
                    chunk = r.read(1 << 20)
                    if not chunk: break
                    f.write(chunk); done += len(chunk)
                    if total:
                        print(f'\r  {os.path.basename(dest)}  '
                              f'{done/2**20:7.1f}/{total/2**20:.1f} MB', end='')
            print(f'\r  {os.path.basename(dest)}  {done/2**20:7.1f} MB  '
                  f'in {time.time()-t0:.0f}s')
            return dest
        except Exception as e:
            print(f'\n  attempt {k+1} failed: {e!r}')
            if os.path.exists(dest): os.remove(dest)
            if k == retries - 1: raise
            time.sleep(5)

for name, url in ASSETS.items():
    fetch(url, f'{OUT}/assets/{name}')


## Vendor pip wheels

So the offline notebook can `pip install --no-index` if the base image is missing anything.

In [ ]:
sh(f'pip download -q -d {OUT}/wheels --no-deps ' + ' '.join(WHEELS) + ' || true')
ws = os.listdir(f'{OUT}/wheels')
print(f'{len(ws)} wheels vendored')
for w in sorted(ws)[:20]: print('  ', w)


## Verify and write the manifest

The manifest is the sentinel notebook 02 looks for when it auto-discovers the mount.

In [ ]:
import torch

man = {'created': time.strftime('%Y-%m-%d %H:%M:%S'),
       'repo_url': REPO_URL, 'repo_branch': REPO_BRANCH, 'assets': {}}

ok = True
for name in ASSETS:
    p = f'{OUT}/assets/{name}'
    n = os.path.getsize(p)
    h = hashlib.sha256(open(p,'rb').read()).hexdigest()[:16]
    man['assets'][name] = {'bytes': n, 'sha256_16': h}
    print(f'  {name:<40} {n/2**20:8.1f} MB  sha256:{h}')
    if n < 1 << 20: ok = False; print('     ^ suspiciously small')

# Prove the checkpoint actually loads with this edm checkout.
sys.path.insert(0, f'{OUT}/third_party/edm')
import pickle
with open(f'{OUT}/assets/edm-cifar10-32x32-cond-vp.pkl','rb') as f:
    net = pickle.load(f)['ema']
print(f"\ncheckpoint loads: {type(net).__name__}, "
      f"{net.img_resolution}x{net.img_resolution}, label_dim={net.label_dim}, "
      f"sigma in [{net.sigma_min}, {net.sigma_max}]")
man['network'] = {'class': type(net).__name__,
                  'img_resolution': int(net.img_resolution),
                  'label_dim': int(net.label_dim)}

json.dump(man, open(f'{OUT}/manifest.json','w'), indent=2)
print('\nmanifest.json written.' if ok else '\nPROBLEM: re-run before using offline.')


## Done

**Save Version -> Save & Run All.** Then in notebook 02, add this notebook's
output via **+ Add Input -> Your Work -> Notebooks**, and set `PREFETCH_NAME`
there to this notebook's slug.


In [ ]:
sh(f'du -sh {OUT}/* | sort -h')
